In [ ]:
from model.train_models import train_evaluate_model
from utils.data_prep import get_clean_combined_data

In [ ]:
xgb_params = {
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "max_delta_step": [0, 1, 5],
    "gamma": [0, 1, 3, 5],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 1, 2],
    "reg_lambda": [1, 5, 10],
    "colsample_bylevel": [0.6, 0.8, 1.0],
}

In [ ]:
winning_config = {
    "include_food": True,
    "include_rain": False,
    "include_text": True,
    "conflict_only": True,
    "k": 1.0,
    "event_col": "sub_event_type",
    "n_splits": 4,
    "use_pca": True,
}

# Locked, tuned hyperparameters from the winning run
# (acled_sub_food_text_conflict_pca_1_4, onset_aupr=0.4214) -
# no search, deterministic every time this cell runs.
winning_xgb_params = {
    "max_depth": 3,
    "min_child_weight": 5,
    "max_delta_step": 0,
    "gamma": 0,
    "learning_rate": 0.01,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 2.0,
    "reg_lambda": 5,
    "colsample_bylevel": 0.6,
}

data_sources = [
    src
    for src, include in zip(
        ["food", "rain", "text"],
        [
            winning_config["include_food"],
            winning_config["include_rain"],
            winning_config["include_text"],
        ],
    )
    if include
]

model_data, predictor_cols = get_clean_combined_data(
    data_sources=data_sources,
    k=winning_config["k"],
    event_col=winning_config["event_col"],
    conflict_only_embeddings=winning_config["conflict_only"],
)

final_params = {
    **winning_xgb_params,
    "k": winning_config["k"],
    "event_col": winning_config["event_col"],
    "n_splits": winning_config["n_splits"],
    "use_pca": winning_config["use_pca"],
}

results, best_params, shap_importance = train_evaluate_model(
    model_data,
    predictor_cols,
    final_params,
    best_params=True,  # skip RandomizedSearchCV entirely - use fixed params
    use_pca=winning_config["use_pca"],
    compute_shap=True,
    shap_sample_size=2000,
)

print(results)
print(shap_importance)